# MathLive input widgets showcase

This notebook explains the **figure-free semantic MathLive layer**. It starts from the user-facing workflow and only looks at the frontend transport snapshot after the widgets are already doing useful work.

What you should learn here:

- how `symbol(...)` relates to SymPy
- how `ExpressionContext` keeps symbols atomic and functions callable
- how `IdentifierInput` and `ExpressionInput` turn synchronized LaTeX/MathJSON state into canonical backend values
- how `transport_manifest()` is a **derived frontend snapshot**, not the primary Python authoring API


## Setup

The notebook adds the repository `src/` folder to `sys.path` by walking upward until it finds `.root`. There is no package installation step and no figure-related setup.


In [7]:
import import_setup

In [8]:
import sympy as sp
from IPython.display import Javascript, Markdown, display

from gu_toolkit import NamedFunction
from gu_toolkit.identifiers import symbol
from gu_toolkit.mathlive import (
    ExpressionContext,
    ExpressionInput,
    IdentifierInput,
    MathJSONParseError,
    mathjson_to_identifier,
)


## 0. Build the semantic context that the widgets should understand

The toolkit `symbol(...)` helper is a thin wrapper around `sp.Symbol(...)`: it validates the symbol name, optionally stores a display-LaTeX override, and forwards normal SymPy assumptions such as `real=True`. Use `symbol()` when you want toolkit-aware validation for one symbol; use `sp.Symbol(...)` for raw SymPy construction; use `sp.symbols(...)` when you want many plain SymPy symbols at once.

`ExpressionContext` then tracks **two semantic categories on purpose**:

- `ctx.symbols` for atomic names such as `velocity`
- `ctx.functions` for callable heads such as `Force`

That separation is why `from_symbols(..., functions=[...])` asks for both categories explicitly. In this notebook we pass `include_named_functions=False` so the tutorial only contains the names registered here, not every `@NamedFunction` that might exist elsewhere in the process. If you already have a SymPy expression, `ExpressionContext.from_expression(...)` can infer the context instead. If you are building it gradually, use `register_symbol(...)` and `register_function(...)`.


In [9]:
@NamedFunction
def Force(x):
    return x

velocity = symbol("velocity")
theta_x_literal = symbol("theta__x")
a_12 = symbol("a_1_2")
x = symbol("x")

ctx = ExpressionContext.from_symbols(
    [velocity, theta_x_literal, a_12, x],
    functions=[Force],
    include_named_functions=False,
)

context_summary = {
    "registered_symbols": list(ctx.symbols),
    "registered_functions": list(ctx.functions),
    "symbol_display": {name: spec.latex_expr for name, spec in ctx.symbols.items()},
    "function_display": {name: spec.latex_head for name, spec in ctx.functions.items()},
}
context_summary


{'registered_symbols': ['velocity', 'theta__x', 'a_1_2', 'x'],
 'registered_functions': ['Force'],
 'symbol_display': {'velocity': '\\mathrm{velocity}',
  'theta__x': '\\mathrm{theta\\_x}',
  'a_1_2': 'a_{1,2}',
  'x': 'x'},
 'function_display': {'Force': '\\operatorname{Force}'}}

`context_summary` above is the friendly Python-side model: symbols and functions are first-class named records. The frontend transport manifest exists too, but it is a *derived* snapshot and we will inspect it later instead of treating raw transport dictionaries as the starting point.


## 1. `IdentifierInput` manual exploration

The first code cell below only displays the widget. If you are running this notebook interactively, edit it manually **or** use the optional JavaScript injection cell that follows. Then re-run the feedback cell to inspect the current canonical identifier plus the supported public widget traits.


In [15]:
identifier_widget = IdentifierInput(
    context=ctx,
    value=r"a_{1,2}",
    aria_label="Identifier demo input",
)
display(identifier_widget)
display(Markdown("Edit the widget manually, or run the optional browser-side injection cell below, then re-run the feedback cell."))


Edit the widget manually, or run the optional browser-side injection cell below, then re-run the feedback cell.

The next code cell is optional and only matters in a live notebook frontend. It uses JavaScript to inject `a_1_2` into the visible widget by targeting the demo field's unique `aria-label`. Under headless `NotebookClient` execution the cell is inert, but it keeps the published notebook honest about the difference between browser-driven edits and kernel-side regression tests.


In [13]:
display(Javascript(r"""
(() => {
  const selector = 'math-field[aria-label="Identifier demo input"], textarea[aria-label="Identifier demo input"]';
  const field = document.querySelector(selector);
  if (!field) {
    console.warn("Identifier demo input not found. Run the display cell above first.");
    return;
  }
  const nextValue = String.raw`a_1_2`;
  try {
    if (typeof field.setValue === "function") {
      field.setValue(nextValue, { silenceNotifications: false });
    } else {
      field.value = nextValue;
    }
  } catch (error) {
    field.value = nextValue;
  }
  field.dispatchEvent(new Event("input", { bubbles: true }));
  field.dispatchEvent(new Event("change", { bubbles: true }));
})();
"""))
display(Markdown("In a live frontend, the browser just injected `a_1_2` into the visible widget. Re-run the next cell to inspect the synced Python state."))


<IPython.core.display.Javascript object>

In a live frontend, the browser just injected `a_1_2` into the visible widget. Re-run the next cell to inspect the synced Python state.

In [16]:
current_identifier = identifier_widget.parse_value()
display(Markdown(f"Current canonical identifier: `{current_identifier}`"))

identifier_state = {
    "value": identifier_widget.value,
    "math_json": identifier_widget.math_json,
    "transport_valid": identifier_widget.transport_valid,
    "transport_errors": identifier_widget.transport_errors,
}
identifier_state


Current canonical identifier: `a_1_2`

{'value': 'a_{1,2}',
 'math_json': None,
 'transport_valid': True,
 'transport_errors': []}

The next cell is the automated **kernel-side** integration check. It deliberately uses a **fresh widget** instead of mutating the visible demo above. That keeps the notebook executable in headless test runs while still leaving the manual browser-driven workflow visible in the earlier cells.


In [17]:
identifier_regression_widget = IdentifierInput(context=ctx, value=r"a_{1,2}")
identifier_regression = {}

assert identifier_regression_widget.parse_value() == "a_1_2"
identifier_regression["latex_text"] = identifier_regression_widget.parse_value()

identifier_regression_widget.value = ""
identifier_regression_widget.math_json = ["Subscript", "a", ["Tuple", 1, 2]]
identifier_regression["mathjson"] = identifier_regression_widget.parse_value()
assert identifier_regression["mathjson"] == "a_1_2"

identifier_regression_widget.value = "x"
identifier_regression["text_after_mathjson"] = identifier_regression_widget.parse_value()
assert identifier_regression["text_after_mathjson"] == "x"

display(Markdown(
    "`IdentifierInput` normalizes both LaTeX text and MathJSON to the same canonical name, "
    "and it falls back to the current text when an older MathJSON payload becomes stale."
))
identifier_regression


`IdentifierInput` normalizes both LaTeX text and MathJSON to the same canonical name, and it falls back to the current text when an older MathJSON payload becomes stale.

{'latex_text': 'a_1_2', 'mathjson': 'a_1_2', 'text_after_mathjson': 'x'}

## 2. `ExpressionInput` keeps known names atomic

Without a context, names like `velocity` could be parsed as a product of single-letter symbols. With a context, the expression widget keeps registered names atomic and keeps registered functions callable.

The rendering boundary is also important: `ctx.render_latex(...)` is a convenience wrapper around SymPy's own `sp.latex(...)` printer. The context contributes semantic symbol/function metadata, but SymPy is still the printer of record.


In [18]:
expression_widget = ExpressionInput(
    context=ctx,
    value=r"\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}",
    aria_label="Expression demo input",
)
display(expression_widget)
display(Markdown("Run the next cell to inspect the parsed SymPy expression and its LaTeX rendering."))


Run the next cell to inspect the parsed SymPy expression and its LaTeX rendering.

In [19]:
parsed_from_text = expression_widget.parse_value()
sympy_latex = sp.latex(parsed_from_text, symbol_names=ctx.symbol_name_map(parsed_from_text))
wrapper_latex = ctx.render_latex(parsed_from_text)

assert parsed_from_text == Force(theta_x_literal) + 2 * velocity
assert wrapper_latex == sympy_latex

display(Markdown(f"Parsed SymPy expression: `{sp.sstr(parsed_from_text)}`"))
display(Markdown(f"SymPy printer output: `${sympy_latex}$`"))

{
    "parsed_expression": parsed_from_text,
    "sympy_latex": sympy_latex,
    "wrapper_latex": wrapper_latex,
}


Parsed SymPy expression: `2*velocity + Force(theta__x)`

SymPy printer output: `$2 \mathrm{velocity} + \operatorname{Force}(\mathrm{theta\_x})$`

{'parsed_expression': 2*velocity + Force(theta__x),
 'sympy_latex': '2 \\mathrm{velocity} + \\operatorname{Force}(\\mathrm{theta\\_x})',
 'wrapper_latex': '2 \\mathrm{velocity} + \\operatorname{Force}(\\mathrm{theta\\_x})'}

## 3. MathJSON integration check

When the MathLive Compute Engine is available, the widget can send a structured MathJSON payload instead of relying only on string parsing. That is the preferred transport **when it still matches the current visible field state**, because it carries more semantic structure and is easier to validate.

Use a fresh widget for the regression below so this transport check does not mutate the earlier text-based teaching example.


In [ ]:
expression_mathjson_widget = ExpressionInput(context=ctx, value="")
expression_mathjson_widget.math_json = [
    "Add",
    ["Force", "theta__x"],
    ["Multiply", "velocity", 2],
]

parsed_from_mathjson = expression_mathjson_widget.parse_value()
assert parsed_from_mathjson == Force(theta_x_literal) + 2 * velocity
assert parsed_from_mathjson == parsed_from_text

parsed_from_mathjson


## 4. `transport_manifest()` is a derived frontend snapshot

At this point the user-facing workflow is already working, so now the low-level transport is easier to place architecturally.

`ctx.transport_manifest(...)` is the JSON-safe payload sent to the frontend. It is **not** the primary Python authoring interface. The small schema worth knowing is:

- top level: `version`, `fieldRole`, `symbols`, `functions`
- symbol entries: `name`, `latex`, plus trigger metadata
- function entries: `name`, `latexHead`, `template`, and optional `arity`

That is enough for debugging or for understanding the browser contract without teaching the raw manifest before the conceptual model.


In [ ]:
manifest = ctx.transport_manifest(field_role="expression")
symbols_by_name = {entry["name"]: entry for entry in manifest["symbols"]}
functions_by_name = {entry["name"]: entry for entry in manifest["functions"]}

assert manifest["fieldRole"] == "expression"
assert symbols_by_name["velocity"]["latex"] == r"\mathrm{velocity}"
assert functions_by_name["Force"]["latexHead"] == r"\operatorname{Force}"
assert functions_by_name["Force"]["template"] == r"\operatorname{Force}(#0)"

{
    "fieldRole": manifest["fieldRole"],
    "velocity_entry": symbols_by_name["velocity"],
    "Force_entry": functions_by_name["Force"],
}


## 5. Some errors are intentional safeguards

A transport spelling such as `theta_x` is ambiguous when no context says whether it should mean a real subscript or a literal underscore inside the atom. In that situation the toolkit **raises** instead of guessing.

This is part of the overall design: fail early at the semantic boundary, not later inside unrelated plotting code.


In [ ]:
try:
    mathjson_to_identifier("theta_x")
except MathJSONParseError as exc:
    ambiguous_message = str(exc)
else:
    raise AssertionError("Expected an ambiguity error for an unregistered single-underscore name.")

assert "ambiguous" in ambiguous_message.lower()
assert mathjson_to_identifier("theta__x", context=ctx) == "theta__x"

ambiguous_message


## 6. Empty MathJSON means “no input”

The frontend may represent an empty field with a sentinel MathJSON payload such as `Nothing`. That payload should behave like missing input, **not** like the number `0` and **not** like a literal identifier named `Nothing` when the text field is empty.


In [ ]:
empty_expression_widget = ExpressionInput(value="")
empty_expression_widget.math_json = "Nothing"

empty_identifier_widget = IdentifierInput(value="")
empty_identifier_widget.math_json = "Nothing"

for widget, role in ((empty_expression_widget, "expression"), (empty_identifier_widget, "identifier")):
    try:
        widget.parse_value()
    except ValueError as exc:
        assert "required" in str(exc)
    else:
        raise AssertionError(f"Expected {role} parsing to reject empty/sentinel MathJSON.")

display(Markdown("Empty/sentinel MathJSON now behaves like missing input instead of parsing as `0` or `Nothing`."))


## Try your own variations

Good experiments after reading this notebook:

- register a new symbol such as `sigma_1` or `sigma__x` and compare the rendered LaTeX
- add a new function name such as `Energy_t` and inspect the transport manifest again
- change the widget text to plain canonical input (`Force(theta__x) + 2*velocity`) and compare it with the explicit LaTeX form
- compare what happens with `theta_x` before and after registering it in the context
